In [1]:
import os
import sys
import math

sys.path.insert(0, os.path.abspath(".."))

import spikecorec as spc
from video_utils import *

MODELS = os.path.join("..", "models")

# Demo 1: Single Cell Example

In [3]:

# simulation parameters

simulation_seconds = 0.5
video_frame_stride = 5

model = str(os.path.join(MODELS, "LEMS_single_cell.xml"))

In [4]:
# run the simulation

engine = spc.SpikeEngine(model)

engine.record_membrane_video(demo_path("single_cell_membrane.spire"),
                             video_frame_stride)
engine.run()
engine.write_recordings()
engine.write_spike_file(demo_path("single_cell_spikes.dat"))

2026-08-28 11:15:18.061 [info] SpikeEngine: stimulus — 1 continuous injections, 0 spike trains
2026-08-28 11:15:18.066 [info] SpikeEngine constructed from ../models/LEMS_single_cell.xml: 1 neurons, 0 edges, 1 cell types, 0 synapse prototypes, dt=0.0001 s, 5000 ticks, spike history 2 rows
2026-08-28 11:15:18.068 [info] record_membrane_video: 1 neurons every 5 ticks -> /Users/aleksimpson/Desktop/projects/spikecorec/build/demos/single_cell_membrane.spire
2026-08-28 11:15:18.069 [info] SpikeEngine: running 5000 ticks at dt=0.0001 s
2026-08-28 11:15:19.287 [info] SpikeEngine: run finished — 50 spikes, mean rate 100.00 Hz, 100.0% of neurons spiked at least once
2026-08-28 11:15:19.287 [info] SpikeEngine: membrane video recording closed
2026-08-28 11:15:19.288 [info] write_spike_file: wrote 50 spikes to /Users/aleksimpson/Desktop/projects/spikecorec/build/demos/single_cell_spikes.dat


In [5]:
# process the output


spike_times, _ = engine.recorded_spikes
measured_interval = float(spike_times[1] - spike_times[0]) if spike_times.size > 1 else 0.0
analytic_interval = 0.005 + 0.010 * math.log(50.0 / 30.0)

print("spikes                  : %d" % spike_times.size)
print("mean firing rate        : %.2f Hz" % engine.mean_firing_rate_hertz())
print("measured interspike gap : %.4f s" % measured_interval)
print("analytic interspike gap : %.4f s" % analytic_interval)
print("relative error          : %.3f %%"
      % (100.0 * abs(measured_interval - analytic_interval) / analytic_interval))
print("final membrane potential: %.2f mV"
      % (1000.0 * engine.read_state_variable(0, "v")))

engine.shutdown()

render_membrane_video(demo_path("single_cell_membrane.spire"),
                      spikes_path=demo_path("single_cell_spikes.dat"),
                      duration=simulation_seconds,
                      trace_neurons=[(0, "the cell")],
                      title="GLIF1 under a 500 pA step")

spikes                  : 50
mean firing rate        : 100.00 Hz
measured interspike gap : 0.0100 s
analytic interspike gap : 0.0101 s
relative error          : 1.071 %
final membrane potential: -70.00 mV
2026-08-28 11:15:25.229 [info] SpikeEngine: shutting down
single_cell_membrane.spire: 1000 frames of 1 neurons -> /Users/aleksimpson/Desktop/projects/spikecorec/build/demos/single_cell_membrane.mp4 (300 rendered at 30 fps)


'/Users/aleksimpson/Desktop/projects/spikecorec/build/demos/single_cell_membrane.mp4'

# Demo 2: Three Cell Chain

In [6]:
simulation_seconds = 0.5
video_frame_stride = 5
model = str(os.path.join(MODELS, "LEMS_three_cell_chain.xml"))

# 0 -> 1 -> 2
topology = [[1], [2], []]

In [7]:
engine = spc.SpikeEngine(str(model), topology, "chainSynapse",
                         connection_weight=1.0,
                         connection_delay_seconds=1e-3)

engine.record_membrane_video(demo_path("three_cell_chain_membrane.spire"),
                             video_frame_stride)
engine.run()
engine.write_recordings()
engine.write_spike_file(demo_path("three_cell_chain_spikes.dat"))

2026-08-28 11:16:40.566 [info] SpikeEngine: topology applied — 3 neurons, 2 edges, weight 1, delay 10 ticks; chainSynapse on 3 cells (2 edges)
2026-08-28 11:16:40.566 [info] WeightMatrix: 1 projections over 2 edges at rank 4 -- the basis reproduces 2 of them, 0 need correcting. Holding 0 weight and 0 delay corrections; 96 bytes basis + 0 bytes corrections against 8 for one float per edge. Worst weight error 0.000e+00.
2026-08-28 11:16:40.566 [info] SpikeEngine: stimulus — 0 continuous injections, 1 spike trains
2026-08-28 11:16:40.567 [info] SpikeEngine constructed from ../models/LEMS_three_cell_chain.xml: 3 neurons, 2 edges, 1 cell types, 1 synapse prototypes, dt=0.0001 s, 5000 ticks, spike history 11 rows
2026-08-28 11:16:40.570 [info] record_membrane_video: 3 neurons every 5 ticks -> /Users/aleksimpson/Desktop/projects/spikecorec/build/demos/three_cell_chain_membrane.spire
2026-08-28 11:16:40.570 [info] SpikeEngine: running 5000 ticks at dt=0.0001 s
2026-08-28 11:16:41.940 [info] Sp

In [8]:
spike_counts = engine.spike_counts_per_neuron
spike_times, spike_neurons = engine.recorded_spikes

print("neurons            : %d" % engine.total_neuron_count)
print("edges              : %d" % engine.weights.total_edge_count)
print("spikes per cell    : A=%d  B=%d  C=%d"
      % (spike_counts[0], spike_counts[1], spike_counts[2]))
print("mean firing rate   : %.2f Hz" % engine.mean_firing_rate_hertz())

engine.shutdown()

render_membrane_video(demo_path("three_cell_chain_membrane.spire"),
                      spikes_path=demo_path("three_cell_chain_spikes.dat"),
                      duration=simulation_seconds,
                      trace_neurons=[(1, "cell B, driven by A"),
                                     (2, "cell C, end of chain")],
                      title="Three-cell chain, A to B to C")

neurons            : 3
edges              : 2
spikes per cell    : A=10  B=10  C=10
mean firing rate   : 20.00 Hz
2026-08-28 11:16:43.755 [info] SpikeEngine: shutting down
three_cell_chain_membrane.spire: 1000 frames of 3 neurons -> /Users/aleksimpson/Desktop/projects/spikecorec/build/demos/three_cell_chain_membrane.mp4 (300 rendered at 30 fps)


'/Users/aleksimpson/Desktop/projects/spikecorec/build/demos/three_cell_chain_membrane.mp4'

# Demo 3: GLIF1 torus

In [9]:
# simulation parameters

video_frame_stride = 5

side_length = 5
simulation_seconds = 0.5
model = str(os.path.join(MODELS, "LEMS_glif1_torus.xml"))
topology = spc.square_torus(side_length)


In [10]:
# Run the simulation

engine = spc.SpikeEngine(model, topology,
                         ["excitatorySynapse", "inhibitorySynapse"],
                         synapse_proportions=[0.8, 0.2],
                         connection_weight=1.0,
                         connection_delay_seconds=1e-3)

engine.record_membrane_video(demo_path("glif1_torus_membrane.spire"),
                             video_frame_stride)
engine.run()
engine.write_recordings()
engine.write_spike_file(demo_path("glif1_torus_spikes.dat"))



2026-08-28 11:21:10.748 [info] SpikeEngine: topology applied — 25 neurons, 100 edges, weight 1, delay 10 ticks; excitatorySynapse on 17 cells (68 edges), inhibitorySynapse on 8 cells (32 edges)
2026-08-28 11:21:10.748 [info] WeightMatrix: 11 projections over 100 edges at rank 4 -- the basis reproduces 100 of them, 0 need correcting. Holding 0 weight and 0 delay corrections; 800 bytes basis + 0 bytes corrections against 400 for one float per edge. Worst weight error 0.000e+00.
2026-08-28 11:21:10.748 [info] SpikeEngine: stimulus — 25 continuous injections, 0 spike trains
2026-08-28 11:21:10.749 [info] SpikeEngine constructed from ../models/LEMS_glif1_torus.xml: 25 neurons, 100 edges, 1 cell types, 2 synapse prototypes, dt=0.0001 s, 5000 ticks, spike history 11 rows
2026-08-28 11:21:10.752 [info] record_membrane_video: 25 neurons every 5 ticks -> /Users/aleksimpson/Desktop/projects/spikecorec/build/demos/glif1_torus_membrane.spire
2026-08-28 11:21:10.752 [info] SpikeEngine: running 5000 

In [11]:
# Render / process the output


# Which synapse each cell's edges carry: 0 excitatory, 1 inhibitory, in the order listed.
# The edges are split into contiguous shares, so a cell at the boundary can carry both (-1).
weights = engine.weights
def synapse_of_cell(cell):
    carried = {weights.get_edge_synapse_prototype(cell, int(target)) for target in weights.get_neighbors(cell)}
    return carried.pop() if len(carried) == 1 else -1
choice = [synapse_of_cell(cell) for cell in range(engine.total_neuron_count)]
spike_counts = engine.spike_counts_per_neuron
excitatory_count = sum(1 for value in choice if value == 0)
inhibitory_count = sum(1 for value in choice if value == 1)

print(f"neurons           : {engine.total_neuron_count} ({excitatory_count} excitatory, {inhibitory_count} inhibitory)")
print(f"edges             : {engine.weights.total_edge_count}")
print(f"spikes            : {int(sum(spike_counts))}")
print(f"mean firing rate  : {engine.mean_firing_rate_hertz():.2f}")
print(f"neurons that fired: {(100.0 * engine.fraction_of_neurons_that_spiked()):.0f}")

def first_active_cell(wanted_choice):
    return next(index for index in range(engine.total_neuron_count)
                if choice[index] == wanted_choice and spike_counts[index] > 0)

excitatory_cell = first_active_cell(0)
inhibitory_cell = first_active_cell(1)
print(f"tracing : excitatory cell {excitatory_cell}, inhibitory cell {inhibitory_cell}")

engine.shutdown()

render_membrane_video(demo_path("glif1_torus_membrane.spire"),
                      spikes_path=demo_path("glif1_torus_spikes.dat"),
                      duration=simulation_seconds,
                      trace_neurons=[(excitatory_cell, f"excitatory cell {excitatory_cell}"),
                                     (inhibitory_cell, f"inhibitory cell {inhibitory_cell}")],
                      title="GLIF1, 5x5 torus, 4:1 excitatory/inhibitory")

neurons           : 25 (17 excitatory, 8 inhibitory)
edges             : 100
spikes            : 285
mean firing rate  : 22.80
neurons that fired: 84
tracing : excitatory cell 0, inhibitory cell 1
2026-08-28 11:21:16.710 [info] SpikeEngine: shutting down
glif1_torus_membrane.spire: 1000 frames of 25 neurons -> /Users/aleksimpson/Desktop/projects/spikecorec/build/demos/glif1_torus_membrane.mp4 (300 rendered at 30 fps)


'/Users/aleksimpson/Desktop/projects/spikecorec/build/demos/glif1_torus_membrane.mp4'

# Demo 4: GLIF5 torus

In [12]:
side_length = 10
simulation_seconds = 0.5
video_frame_stride = 5

model = str(os.path.join(MODELS, "LEMS_glif5_torus.xml"))

topology = spc.square_torus(side_length)

In [13]:
engine = spc.SpikeEngine(model, topology,
                         ["excitatorySynapse", "inhibitorySynapse"],
                         synapse_proportions=[0.8, 0.2],
                         connection_weight=1.0,
                         connection_delay_seconds=1e-3)

engine.record_membrane_video(demo_path("glif5_torus_membrane.spire"),
                             video_frame_stride)
engine.run()
engine.write_recordings()
engine.write_spike_file(demo_path("glif5_torus_spikes.dat"))

2026-08-28 11:24:19.523 [info] SpikeEngine: topology applied — 100 neurons, 400 edges, weight 1, delay 10 ticks; excitatorySynapse on 73 cells (292 edges), inhibitorySynapse on 27 cells (108 edges)
2026-08-28 11:24:19.524 [info] WeightMatrix: 37 projections over 400 edges at rank 4 -- the basis reproduces 400 of them, 0 need correcting. Holding 0 weight and 0 delay corrections; 3200 bytes basis + 0 bytes corrections against 1600 for one float per edge. Worst weight error 0.000e+00.
2026-08-28 11:24:19.524 [info] SpikeEngine: stimulus — 100 continuous injections, 0 spike trains
2026-08-28 11:24:19.524 [info] SpikeEngine constructed from ../models/LEMS_glif5_torus.xml: 100 neurons, 400 edges, 1 cell types, 2 synapse prototypes, dt=0.0001 s, 5000 ticks, spike history 11 rows
2026-08-28 11:24:19.527 [info] record_membrane_video: 100 neurons every 5 ticks -> /Users/aleksimpson/Desktop/projects/spikecorec/build/demos/glif5_torus_membrane.spire
2026-08-28 11:24:19.528 [info] SpikeEngine: runn

In [14]:
# Which synapse each cell's edges carry: 0 excitatory, 1 inhibitory, in the order listed.
# The edges are split into contiguous shares, so a cell at the boundary can carry both (-1).
weights = engine.weights
def synapse_of_cell(cell):
    carried = {weights.get_edge_synapse_prototype(cell, int(target)) for target in weights.get_neighbors(cell)}
    return carried.pop() if len(carried) == 1 else -1
choice = [synapse_of_cell(cell) for cell in range(engine.total_neuron_count)]
spike_counts = engine.spike_counts_per_neuron
excitatory_count = sum(1 for value in choice if value == 0)
inhibitory_count = sum(1 for value in choice if value == 1)

print("neurons            : %d (%d excitatory, %d inhibitory)"
      % (engine.total_neuron_count, excitatory_count, inhibitory_count))
print("edges              : %d" % engine.weights.total_edge_count)
print("spikes             : %d" % int(sum(spike_counts)))
print("mean firing rate   : %.2f Hz" % engine.mean_firing_rate_hertz())
print("neurons that fired : %.0f %%"
      % (100.0 * engine.fraction_of_neurons_that_spiked()))

spike_times, spike_neurons = engine.recorded_spikes
busiest = max(range(engine.total_neuron_count), key=lambda index: spike_counts[index])
own_times = [time for time, neuron in zip(spike_times, spike_neurons)
             if neuron == busiest]
if len(own_times) > 2:
    first_interval = own_times[1] - own_times[0]
    last_interval = own_times[-1] - own_times[-2]
    print("adaptation         : cell %d went from %.1f ms to %.1f ms between spikes "
          "(%.2fx)" % (busiest, 1000.0 * first_interval, 1000.0 * last_interval,
                       last_interval / first_interval))

def first_active_cell(wanted_choice):
    return next(index for index in range(engine.total_neuron_count)
                if choice[index] == wanted_choice and spike_counts[index] > 0)

excitatory_cell = first_active_cell(0)
inhibitory_cell = first_active_cell(1)
print("tracing            : excitatory cell %d, inhibitory cell %d"
      % (excitatory_cell, inhibitory_cell))

engine.shutdown()

render_membrane_video(demo_path("glif5_torus_membrane.spire"),
                      spikes_path=demo_path("glif5_torus_spikes.dat"),
                      duration=simulation_seconds,
                      trace_neurons=[(excitatory_cell, "excitatory cell %d" % excitatory_cell),
                                     (inhibitory_cell, "inhibitory cell %d" % inhibitory_cell)],
                      title="GLIF5, 10x10 torus, adapting cells")



neurons            : 100 (73 excitatory, 27 inhibitory)
edges              : 400
spikes             : 477
mean firing rate   : 9.54 Hz
neurons that fired : 93 %
adaptation         : cell 77 went from 12.1 ms to 36.0 ms between spikes (2.98x)
tracing            : excitatory cell 0, inhibitory cell 1
2026-08-28 11:24:24.203 [info] SpikeEngine: shutting down
glif5_torus_membrane.spire: 1000 frames of 100 neurons -> /Users/aleksimpson/Desktop/projects/spikecorec/build/demos/glif5_torus_membrane.mp4 (300 rendered at 30 fps)


'/Users/aleksimpson/Desktop/projects/spikecorec/build/demos/glif5_torus_membrane.mp4'